# Perceptron

Classificador linear binario, ancestral das redes neurais.

Faz updates online sempre que erra: a fronteira se move em direcao
as amostras mal classificadas. Se os dados forem linearmente separaveis,
converge; caso contrario, oscila (entao limitamos com `n_iters`).

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn import datasets
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score

In [ ]:
class Perceptron:
    """
    Perceptron binario de Rosenblatt.

    Regra de atualizacao online:
        w <- w + lr * (y - y_hat) * x
        b <- b + lr * (y - y_hat)

    Converge em numero finito de passos se os dados forem linearmente separaveis.
    """

    def __init__(self, lr=0.01, n_iters=1000, random_state=None):
        if lr <= 0:
            raise ValueError("lr must be positive.")
        if n_iters <= 0:
            raise ValueError("n_iters must be positive.")

        self.lr = lr
        self.n_iters = n_iters
        self.random_state = random_state

        self.weights = None
        self.bias = 0.0
        self.classes_ = None
        self.errors_ = []

    @staticmethod
    def _step(z):
        return np.where(z >= 0, 1, 0)

    def fit(self, X, y):
        X = np.asarray(X, dtype=float)
        y = np.asarray(y).ravel()
        if X.ndim != 2:
            raise ValueError("X must be 2D.")

        self.classes_ = np.unique(y)
        if len(self.classes_) != 2:
            raise ValueError("Perceptron supports binary classification only.")

        # mapeia para {0, 1}
        y01 = (y == self.classes_[1]).astype(int)

        rng = np.random.default_rng(self.random_state)
        self.weights = rng.normal(0, 0.01, size=X.shape[1])
        self.bias = 0.0
        self.errors_ = []

        for _ in range(self.n_iters):
            errors = 0
            for xi, target in zip(X, y01):
                pred = self._step(np.dot(xi, self.weights) + self.bias)
                update = self.lr * (target - pred)
                if update != 0.0:
                    self.weights += update * xi
                    self.bias += update
                    errors += 1
            self.errors_.append(errors)
            if errors == 0:
                break

        return self

    def predict(self, X):
        if self.weights is None:
            raise ValueError("Call fit() before predict().")
        X = np.asarray(X, dtype=float)
        if X.ndim == 1:
            X = X.reshape(1, -1)
        y01 = self._step(X @ self.weights + self.bias)
        return np.where(y01 == 1, self.classes_[1], self.classes_[0])

    def score(self, X, y):
        return np.mean(self.predict(X) == np.asarray(y))

In [ ]:
X, y = datasets.make_blobs(
    n_samples=200, centers=2, n_features=2,
    cluster_std=1.2, random_state=42
)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

p = Perceptron(lr=0.1, n_iters=100, random_state=42)
p.fit(X_train, y_train)
print("Accuracy:", accuracy_score(y_test, p.predict(X_test)))
print("Epochs ate convergir:", len(p.errors_))

In [ ]:
# fronteira de decisao
xx, yy = np.meshgrid(
    np.linspace(X[:, 0].min() - 1, X[:, 0].max() + 1, 200),
    np.linspace(X[:, 1].min() - 1, X[:, 1].max() + 1, 200),
)
grid = np.c_[xx.ravel(), yy.ravel()]
Z = p.predict(grid).reshape(xx.shape)

plt.figure(figsize=(7, 5))
plt.contourf(xx, yy, Z, alpha=0.3, cmap="coolwarm")
plt.scatter(X[:, 0], X[:, 1], c=y, edgecolor="k", cmap="coolwarm")
plt.title("Perceptron decision boundary")
plt.show()